# QLoRA GPU Smoke Test — Pre-Flight Check Before the Production GPU

> Verifies the full QLoRA fine-tuning pipeline on a free Kaggle GPU (T4/P100) with a real instruct model (Qwen2.5-1.5B-Instruct): 4-bit NF4 loading, chat-template + prompt-masked SFT, VRAM/throughput measurement, adapter save -> reload. Every check is an `assert`, so a green run means the pipeline is ready to scale up on a production CUDA box.

Companion docs in [ML_report](https://github.com/morimori0456/ML_report): [finetuning_methods_survey.md](https://github.com/morimori0456/ML_report/blob/main/llm/finetuning_methods_survey.md) (method map; this notebook is the GPU leg of its CPU experiment) and [lora_qlora.md](https://github.com/morimori0456/ML_report/blob/main/llm/lora_qlora.md) (NF4/QLoRA internals).

**What transfers to the production GPU**: everything in this pipeline (data encoding, masking, QLoRA config, adapter workflow). **What does not**: throughput numbers (T4/P100 are far slower than A100/H100), and fp16 — on Ampere or newer switch `compute_dtype` to bf16.

**Provenance**: the outputs below are from an actual run on Kaggle (2x Tesla T4, kernel `qlora-gpu-smoke`, 2026-07-13), driven headlessly via the Kaggle API. To rerun: `kaggle kernels push -p <dir-with-this-notebook-and-kernel-metadata> --accelerator NvidiaTeslaT4`. All verification checks passed (see final cell).

In [1]:
# --- 1. Environment: GPU, precision support, library versions ---
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout)

assert torch.cuda.is_available(), "No CUDA GPU -- enable GPU accelerator in Kaggle settings"
DEV = torch.device("cuda")
cc = torch.cuda.get_device_capability()
# Decide bf16 by compute capability, NOT torch.cuda.is_bf16_supported() -- the latter
# returns True on pre-Ampere cards (software emulation), which silently kills performance
# and breaks bitsandbytes. Hardware bf16 = Ampere (8.0) and newer.
BF16_OK = cc >= (8, 0)
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print(f"GPU: {torch.cuda.get_device_name()} | compute capability {cc} | "
      f"hardware bf16: {BF16_OK} -> compute dtype {COMPUTE_DTYPE}")
assert cc >= (7, 0), (f"compute capability {cc} < 7.0: current PyTorch/bitsandbytes wheels "
                      "do not support this GPU (e.g. Kaggle P100). Request a T4 instead.")
# T4 (7.5) -> fp16 here; on A100/H100 (8.0/9.0) this flips to bf16 automatically

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5
Tesla T4, 15360 MiB, 7.5

GPU: Tesla T4 | compute capability (7, 5) | hardware bf16: False -> compute dtype torch.float16


In [2]:
%pip install -q -U transformers peft bitsandbytes accelerate
import transformers, peft, bitsandbytes, accelerate
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | bitsandbytes {bitsandbytes.__version__} | "
      f"accelerate {accelerate.__version__}")
# Pin these versions when moving to the production environment

Note: you may need to restart the kernel to use updated packages.
torch 2.10.0+cu128 | transformers 5.13.0 | peft 0.19.1 | bitsandbytes 0.49.2 | accelerate 1.14.0


In [3]:
# --- 2. Load the base model in 4-bit NF4 (the QLoRA recipe) ---
import time, gc, random
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"      # not gated; ~3GB download
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # quantile-optimal 4-bit float
    bnb_4bit_use_double_quant=True,       # quantize the quantization constants
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb_cfg,
                                             device_map={"": 0})
load_time = time.time() - t0
vram_load = torch.cuda.memory_allocated() / 1e9
print(f"loaded in {load_time:.0f}s | VRAM after load: {vram_load:.2f} GB "
      f"(1.5B params in 4-bit ~= 1.1 GB + overhead)")
assert vram_load < 4.0, "4-bit load should be well under 4 GB for a 1.5B model" 

loaded in 25s | VRAM after load: 1.15 GB (1.5B params in 4-bit ~= 1.1 GB + overhead)


## 3. Task and data — same experiment as the CPU notebook, instruct-model grade

Identical slot-filling task to [finetuning_methods_survey.ipynb](https://github.com/morimori0456/ML_report/blob/main/llm/finetuning_methods_survey.ipynb), but encoded the production way: through the model's **chat template** (`apply_chat_template`), with labels masked over everything except the assistant response (prompt masking). Template mismatch between training and inference is the #1 SFT bug in practice — encoding both through the same API eliminates it by construction.

In [4]:
SKIES = ["sunny", "cloudy", "rainy", "foggy", "windy", "snowy"]
INSTRUCTION = "Convert the sensor data into a one-sentence weather report."

def make_example(t, temp, sky):
    user = f"{INSTRUCTION}\ndata: time={t} temp={temp} sky={sky}"
    assistant = f"At {t} it is {temp} degrees and {sky}."
    return user, assistant

def build_data(n_train=160, n_eval=24):
    combos = [(f"{h:02d}:{m:02d}", temp, sky)
              for h in range(0, 24, 2) for m in (0, 30)
              for temp in range(0, 35, 5) for sky in SKIES]
    random.shuffle(combos)                      # split by combination
    return ([make_example(*c) for c in combos[:n_train]],
            [make_example(*c) for c in combos[n_train:n_train + n_eval]])

train_data, eval_data = build_data()

def ids_of(x):
    """transformers >=5 returns BatchEncoding from apply_chat_template; 4.x returns a list."""
    return x["input_ids"] if hasattr(x, "keys") else x

def encode_batch(pairs):
    """Chat-template encoding with prompt masking: loss only on assistant tokens."""
    seqs = []
    for user, assistant in pairs:
        msgs = [{"role": "user", "content": user}]
        prefix = ids_of(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True))
        full = ids_of(tok.apply_chat_template(msgs + [{"role": "assistant", "content": assistant}],
                                              tokenize=True))
        assert full[:len(prefix)] == prefix, "generation prompt is not a prefix of the full chat"
        labels = [-100] * len(prefix) + full[len(prefix):]   # EOS (<|im_end|>) included by template
        seqs.append((full, labels))
    max_len = max(len(s) for s, _ in seqs)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.tensor([s + [pad] * (max_len - len(s)) for s, _ in seqs])
    lab = torch.tensor([l + [-100] * (max_len - len(l)) for _, l in seqs])
    attn = torch.tensor([[1] * len(s) + [0] * (max_len - len(s)) for s, _ in seqs])
    return ids.to(DEV), lab.to(DEV), attn.to(DEV)

ids, lab, attn = encode_batch(train_data[:2])
n_resp = (lab[0] != -100).sum().item()
print(f"encoded seq len {ids.shape[1]} | response tokens with loss: {n_resp}")
print(tok.decode(ids[0][attn[0].bool()]))
assert n_resp > 5, "prompt masking produced no response tokens" 

encoded seq len 76 | response tokens with loss: 17
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Convert the sensor data into a one-sentence weather report.
data: time=02:00 temp=0 sky=cloudy<|im_end|>
<|im_start|>assistant
At 02:00 it is 0 degrees and cloudy.<|im_end|>


In [5]:
# --- 4. QLoRA fine-tune: LoRA adapters on the 4-bit base ---
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig, TaskType

model = prepare_model_for_kbit_training(model)     # norm layers to fp32, enable grads on inputs
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],   # all linear modules
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

STEPS, BATCH, LR = 100, 8, 2e-4
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
rng = random.Random(0)
torch.cuda.reset_peak_memory_stats()
model.train()
losses, t0 = [], time.time()
for step in range(STEPS):
    ids, lab, attn = encode_batch(rng.sample(train_data, BATCH))
    out = model(input_ids=ids, attention_mask=attn, labels=lab)
    out.loss.backward(); opt.step(); opt.zero_grad()
    losses.append(out.loss.item())
    if (step + 1) % 20 == 0:
        print(f"step {step+1:3d}/{STEPS} | loss {np.mean(losses[-10:]):.4f} | "
              f"{(time.time()-t0)/(step+1):.2f} s/step")
train_time = time.time() - t0
vram_peak = torch.cuda.max_memory_allocated() / 1e9
print(f"\ntrained {STEPS} steps in {train_time:.0f}s | peak VRAM {vram_peak:.2f} GB")
assert losses[-1] < losses[0] / 3, "loss did not drop -- pipeline broken"
assert vram_peak < 14.0, "should fit a 16GB card with headroom" 

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
step  20/100 | loss 0.0003 | 0.94 s/step
step  40/100 | loss 0.0000 | 0.92 s/step
step  60/100 | loss 0.0000 | 0.92 s/step
step  80/100 | loss 0.0000 | 0.92 s/step
step 100/100 | loss 0.0000 | 0.92 s/step

trained 100 steps in 92s | peak VRAM 3.47 GB


In [6]:
# --- 5. Evaluate: greedy decode, exact match on held-out slot combinations ---
@torch.no_grad()
def exact_match(m, data):
    m.eval()
    hits, gens = 0, []
    for user, assistant in data:
        msgs = [{"role": "user", "content": user}]
        prefix = ids_of(tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True))
        enc = torch.tensor([prefix], device=DEV)
        out = m.generate(enc, attention_mask=torch.ones_like(enc),
                         max_new_tokens=32, do_sample=False,
                         pad_token_id=tok.eos_token_id)
        gen = tok.decode(out[0][enc.shape[1]:], skip_special_tokens=True).strip()
        gens.append(gen)
        hits += int(gen == assistant or gen.startswith(assistant))
    return hits / len(data), gens

t0 = time.time()
em, gens = exact_match(model, eval_data)
print(f"exact match on {len(eval_data)} held-out combos: {em:.2f} ({time.time()-t0:.0f}s)")
print(f"sample: {gens[0]!r}")
assert em >= 0.8, f"exact match {em:.2f} below 0.8 -- check masking/template/lr" 

exact match on 24 held-out combos: 1.00 (48s)
sample: 'At 14:30 it is 5 degrees and foggy.'


In [7]:
# --- 6. Adapter save -> fresh reload (the deployment path) ---
import os
from peft import PeftModel

ADAPTER_DIR = "/kaggle/working/qlora_adapter"
model.save_pretrained(ADAPTER_DIR)
sz = sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1e6
print(f"adapter saved: {sz:.1f} MB (vs ~3GB base model)")

del model, opt; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb_cfg,
                                            device_map={"": 0})
reloaded = PeftModel.from_pretrained(base, ADAPTER_DIR)
em2, gens2 = exact_match(reloaded, eval_data[:8])
print(f"reloaded adapter exact match (8 samples): {em2:.2f} | sample: {gens2[0]!r}")
assert em2 >= 0.75, "reloaded adapter lost the fine-tuned behavior" 

adapter saved: 73.9 MB (vs ~3GB base model)
reloaded adapter exact match (8 samples): 1.00 | sample: 'At 14:30 it is 5 degrees and foggy.'


In [8]:
print("=" * 60)
print("ALL CHECKS PASSED — pipeline verified on", torch.cuda.get_device_name())
print("=" * 60)
print(f"""
Measured on this run:
  4-bit load VRAM      : {vram_load:.2f} GB
  training peak VRAM   : {vram_peak:.2f} GB (batch {BATCH}, r=16, all linear)
  throughput           : {train_time/STEPS:.2f} s/step -> reference only, not transferable
  eval exact match     : {em:.2f}
  adapter artifact     : {sz:.1f} MB

Before running on the production GPU:
  1. compute_dtype flips to bf16 automatically on Ampere+ (BF16_OK above)
  2. pin the library versions printed in cell 2
  3. re-tune batch size / grad accumulation to the larger VRAM
  4. consider flash_attention_2 (unavailable on T4/P100, big win on A100/H100)
  5. for >1 epoch of real data, add packing + eval-loss early stopping
""")

ALL CHECKS PASSED — pipeline verified on Tesla T4

Measured on this run:
  4-bit load VRAM      : 1.15 GB
  training peak VRAM   : 3.47 GB (batch 8, r=16, all linear)
  throughput           : 0.92 s/step -> reference only, not transferable
  eval exact match     : 1.00
  adapter artifact     : 73.9 MB

Before running on the production GPU:
  1. compute_dtype flips to bf16 automatically on Ampere+ (BF16_OK above)
  2. pin the library versions printed in cell 2
  3. re-tune batch size / grad accumulation to the larger VRAM
  4. consider flash_attention_2 (unavailable on T4/P100, big win on A100/H100)
  5. for >1 epoch of real data, add packing + eval-loss early stopping


## Takeaways

1. **The entire QLoRA pipeline is verified end-to-end**: NF4 4-bit load, `prepare_model_for_kbit_training`, LoRA on all linear modules, chat-template encoding with prompt masking, greedy-decode evaluation, adapter save/reload. These components transfer unchanged to the production GPU.
2. **fp16 vs bf16 is the one silent difference between free-tier and production cards.** T4/P100 force fp16 compute; the dtype flag here auto-detects, so the same notebook runs correctly on Ampere+ with bf16.
3. **VRAM numbers give the scaling intuition**: a 1.5B model trains in ~a few GB in 4-bit — consistent with 7B fitting a 16GB card and 70B fitting 48GB, per the QLoRA paper.
4. **Throughput on free GPUs is a smoke-test number, not a plan.** Use it only to confirm the loop runs; budget production time on the production card.